# Pipeline 2 – Transfermarkt: Bronze ➜ Silver (incremental by snapshot)

**Input (bronze):** `s3://<bucket>/bronze/transfermarkt/<table>/snapshot_date=YYYY-MM-DD/<table>.csv`

**Output (silver, Parquet, partitioned by `snapshot_date`):**

| table | grain | notes |
|---|---|---|
| `players` | 1 row / player | typed, trimmed, normalised name for matching |
| `clubs`, `competitions` | 1 row / club, competition | lookup dimensions |
| `player_valuations` | 1 row / player / valuation date | + window features (12/24-month trajectory) |
| `transfers` | 1 row / transfer | fee parsed, future-dated "planned" moves flagged |

**Incremental logic:** the Kaggle dataset is a *full* refresh each time, so every drop lands in a new
`snapshot_date=` folder. The job only processes snapshots newer than the latest one already in silver
and skips entirely (cheap no-op) when nothing new has landed.

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

In [ ]:
# --- Glue boilerplate: Spark + Glue contexts and job parameters ---
import sys
import time
import json
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

# Job parameters are passed by the Workflow trigger (--BUCKET ...); interactive runs fall back to defaults
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET"])
except Exception:
    args = {"JOB_NAME": "pl_02_transfermarkt_bronze_to_silver", "BUCKET": "football-analytics-moneyball-<ACCOUNT_ID>"}
job.init(args["JOB_NAME"], args)

In [ ]:
# --- Configuration ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = f"s3://{args['BUCKET']}/bronze/transfermarkt"
SILVER = f"s3://{args['BUCKET']}/silver/transfermarkt"
PIPELINE = "pl_02_transfermarkt"
TABLES = ["competitions", "clubs", "players", "player_valuations", "transfers"]

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
RUN_START = time.time()

In [ ]:
# --- Helpers: storage I/O + observability ---
def list_paths(glob_pattern):
    """List files matching a glob using the Hadoop FS API (works for s3:// and local paths)."""
    path = spark._jvm.org.apache.hadoop.fs.Path(glob_pattern)
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    statuses = fs.globStatus(path) or []
    return [s.getPath().toString() for s in statuses]


def write_table(df, path, partition_cols, mode):
    """Write a Parquet table (snappy) partitioned by the given columns."""
    df.write.mode(mode).partitionBy(*partition_cols).parquet(path)


def emit_metrics(metrics):
    """Structured log line (lands in CloudWatch Logs) + custom CloudWatch metrics."""
    print(json.dumps({"pipeline": PIPELINE, **metrics}))
    try:
        import boto3
        boto3.client("cloudwatch").put_metric_data(
            Namespace="FootballAnalytics/Pipelines",
            MetricData=[
                {"MetricName": k, "Value": float(v), "Unit": "Count",
                 "Dimensions": [{"Name": "Pipeline", "Value": PIPELINE}]}
                for k, v in metrics.items() if isinstance(v, (int, float))
            ],
        )
    except Exception as exc:  # metrics must never fail the pipeline
        print(f"CloudWatch metrics not published: {exc}")

## 1. Find snapshots that have not been processed yet

In [ ]:
def snapshot_of(path):
    """Extract 'YYYY-MM-DD' from a .../snapshot_date=YYYY-MM-DD/... path."""
    return path.split("snapshot_date=")[1].split("/")[0]


# A snapshot is complete only when all 5 tables have landed for that date
landed = {}
for table in TABLES:
    for p in list_paths(f"{BRONZE}/{table}/snapshot_date=*/*.csv"):
        landed.setdefault(snapshot_of(p), set()).add(table)
complete = sorted(d for d, t in landed.items() if t == set(TABLES))

done = sorted({snapshot_of(p) for p in list_paths(f"{SILVER}/players/snapshot_date=*/*.parquet")})
last_done = done[-1] if done else ""
new_snapshots = [d for d in complete if d > last_done]
print(f"Complete snapshots in bronze: {complete} | already in silver: {last_done or 'none'} | to process: {new_snapshots}")

In [ ]:
def read_csv(table, snapshot):
    """Read one bronze CSV. All columns arrive as strings; we cast explicitly in the transforms."""
    return (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
            .csv(f"{BRONZE}/{table}/snapshot_date={snapshot}/{table}.csv"))


def clean_str(c):
    """Trim whitespace and turn empty strings into NULL."""
    return F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))


# The source mixes spellings of the same country (e.g. 1,392 "Turkey" vs 907 "Türkiye" players)
COUNTRY_ALIASES = {"Türkiye": "Turkey"}


def standard_country(c):
    """Clean a country column and collapse known aliases to one spelling."""
    col = clean_str(c)
    for alias, standard in COUNTRY_ALIASES.items():
        col = F.when(col == alias, standard).otherwise(col)
    return col


def normalise_name(c):
    """Lower-case, strip accents and punctuation – the join key used for identity resolution in gold."""
    accents = "áàâãäåāăąçćčďđéèêëēėęěğíìîïıīłńñňóòôõöøōőřśšşșťțúùûüůūűýÿžźżß"
    plain   = "aaaaaaaaacccddeeeeeeeegiiiiiilnnnoooooooorssssttuuuuuuuyyzzzs"
    return F.trim(F.regexp_replace(F.translate(F.lower(F.col(c)), accents, plain), r"[^a-z ]", " ")) \
        .alias("name_normalised")

## 2. Transform each snapshot

In [ ]:
for snap in new_snapshots:
    snap_col = F.lit(snap).cast("date").alias("snapshot_date")

    # --- players: the main dimension ---------------------------------------------------------
    players = read_csv("players", snap).select(
        F.col("player_id").cast("long"),
        clean_str("name").alias("player_name"),
        clean_str("first_name").alias("first_name"),
        clean_str("last_name").alias("last_name"),
        F.regexp_replace(F.trim("name"), r"\s+", " ").alias("_name"),
        F.to_date(F.substring("date_of_birth", 1, 10)).alias("date_of_birth"),
        standard_country("country_of_citizenship").alias("country_of_citizenship"),
        standard_country("country_of_birth").alias("country_of_birth"),
        clean_str("position").alias("position"),
        clean_str("sub_position").alias("sub_position"),
        F.lower(clean_str("foot")).alias("foot"),
        F.col("height_in_cm").cast("int").alias("height_in_cm"),
        F.to_date(F.substring("contract_expiration_date", 1, 10)).alias("contract_expiration_date"),
        clean_str("agent_name").alias("agent_name"),
        F.col("current_club_id").cast("long").alias("current_club_id"),
        clean_str("current_club_name").alias("current_club_name"),
        clean_str("current_club_domestic_competition_id").alias("current_competition_id"),
        F.col("last_season").cast("int").alias("last_season"),
        F.col("international_caps").cast("int").alias("international_caps"),
        F.col("market_value_in_eur").cast("double").cast("long").alias("market_value_eur"),
        F.col("highest_market_value_in_eur").cast("double").cast("long").alias("highest_market_value_eur"),
        clean_str("url").alias("url"),
    ).withColumn("name_normalised", normalise_name("_name")).drop("_name") \
     .where(F.col("player_id").isNotNull()) \
     .dropDuplicates(["player_id"]) \
     .select("*", snap_col)

    # --- clubs / competitions: small lookups ------------------------------------------------
    clubs = read_csv("clubs", snap).select(
        F.col("club_id").cast("long"), clean_str("name").alias("club_name"),
        clean_str("domestic_competition_id").alias("competition_id"),
        F.col("squad_size").cast("int"), F.col("average_age").cast("double"),
        F.col("last_season").cast("int"), snap_col)

    competitions = read_csv("competitions", snap).select(
        clean_str("competition_id").alias("competition_id"),
        F.initcap(F.regexp_replace("name", "-", " ")).alias("competition_name"),
        clean_str("type").alias("type"), clean_str("sub_type").alias("sub_type"),
        clean_str("country_name").alias("country_name"), snap_col)

    # --- player_valuations: history + window-based trajectory features -------------------------
    vals = (read_csv("player_valuations", snap).select(
                F.col("player_id").cast("long"),
                F.to_date("date").alias("valuation_date"),
                F.col("market_value_in_eur").cast("double").cast("long").alias("market_value_eur"),
                F.col("current_club_id").cast("long").alias("club_id"),
                clean_str("current_club_name").alias("club_name"),
                clean_str("player_club_domestic_competition_id").alias("competition_id"))
            .where(F.col("player_id").isNotNull() & F.col("valuation_date").isNotNull()
                   & (F.col("market_value_eur") > 0))
            .dropDuplicates(["player_id", "valuation_date"]))

    # Days since epoch lets us use rangeBetween for "value N months ago" look-backs
    w = Window.partitionBy("player_id").orderBy("day_num")
    vals = (vals.withColumn("day_num", F.datediff("valuation_date", F.lit("1970-01-01")))
            .withColumn("previous_value_eur", F.lag("market_value_eur").over(w))
            .withColumn("value_12m_ago_eur", F.first("market_value_eur").over(w.rangeBetween(-365, -300)))
            .withColumn("value_24m_ago_eur", F.first("market_value_eur").over(w.rangeBetween(-730, -640)))
            .withColumn("peak_value_to_date_eur", F.max("market_value_eur").over(w.rowsBetween(Window.unboundedPreceding, 0)))
            .withColumn("valuation_seq", F.row_number().over(w))
            .withColumn("is_latest", F.row_number().over(Window.partitionBy("player_id").orderBy(F.desc("day_num"))) == 1)
            .drop("day_num")
            .select("*", snap_col))

    # --- transfers: parse fees, flag future-dated (pre-agreed) moves ---------------------------
    transfers = read_csv("transfers", snap).select(
        F.col("player_id").cast("long"),
        clean_str("player_name").alias("player_name"),
        F.to_date("transfer_date").alias("transfer_date"),
        clean_str("transfer_season").alias("transfer_season"),
        F.col("from_club_id").cast("long"), clean_str("from_club_name").alias("from_club_name"),
        F.col("to_club_id").cast("long"), clean_str("to_club_name").alias("to_club_name"),
        F.col("transfer_fee").cast("double").alias("transfer_fee_eur"),
        F.col("market_value_in_eur").cast("double").alias("market_value_at_transfer_eur"),
    ).withColumn("is_future_dated", F.col("transfer_date") > F.lit(snap).cast("date")) \
     .dropDuplicates(["player_id", "transfer_date", "from_club_id", "to_club_id"]) \
     .select("*", snap_col)

    # --- Data-quality gates --------------------------------------------------------------------
    # NB: no .cache() here - a cached players frame produced a wrong anti-join count on Spark 3.5.4
    n_players = players.count()
    dq = {
        "snapshot": snap,
        "players_rows": n_players,
        "players_null_name": players.where(F.col("player_name").isNull()).count(),
        "players_null_dob_pct": round(players.where(F.col("date_of_birth").isNull()).count() / max(n_players, 1) * 100, 2),
        "valuation_rows": vals.count(),
        "valuation_orphans": vals.select("player_id").distinct()
            .join(players.select("player_id"), "player_id", "left_anti").count(),
        "transfer_rows": transfers.count(),
    }
    assert n_players > 0, "players.csv is empty"
    assert dq["players_null_name"] == 0, "Players without a name"
    emit_metrics(dq)

    # --- Write (dynamic overwrite = re-running a snapshot is idempotent) -----------------------
    for name, df in [("competitions", competitions), ("clubs", clubs), ("player_valuations", vals),
                     ("transfers", transfers), ("players", players)]:   # players last = watermark
        write_table(df, f"{SILVER}/{name}", ["snapshot_date"], "overwrite")

In [ ]:
# --- Run summary + commit ---
emit_metrics({"duration_seconds": round(time.time() - RUN_START, 1), "snapshots_processed": len(new_snapshots)})
job.commit()